<a href="https://colab.research.google.com/github/carols-rodrigues/fastqc_script_GA055/blob/main/fastqc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%bash

apt-get -qq update

# Java
apt-get install -y default-jre

# FastQC
apt-get install -y fastqc

# Cutadapt
pip install cutadapt

# Trim Galore
pip install trim-galore

# SRA Toolkit
apt-get install -y sra-toolkit

# MultiQC
pip install multiqc

# Compressão (opcional)
apt-get install -y pigz

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  at-spi2-core default-jre-headless fonts-dejavu-core fonts-dejavu-extra
  gsettings-desktop-schemas libatk-bridge2.0-0 libatk-wrapper-java
  libatk-wrapper-java-jni libatk1.0-0 libatk1.0-data libatspi2.0-0
  libxcomposite1 libxtst6 libxxf86dga1 openjdk-11-jre openjdk-11-jre-headless
  session-migration x11-utils
Suggested packages:
  libnss-mdns fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  | fonts-wqy-zenhei fonts-indic mesa-utils
The following NEW packages will be installed:
  at-spi2-core default-jre default-jre-headless fonts-dejavu-core
  fonts-dejavu-extra gsettings-desktop-schemas libatk-bridge2.0-0
  libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0 libatk1.0-data
  libatspi2.0-0 libxcomposite1 libxtst6 libxxf86dga1 openjdk-11-jre
  openjdk-11-jre-headless session-migration x11-utils
0 upgraded, 19 newly installed, 0 to r

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
ERROR: Could not find a version that satisfies the requirement trim-galore (from versions: none)
ERROR: No matching distribution found for trim-galore


In [ ]:
%%writefile pipeline.sh
#!/bin/bash

set -euo pipefail

###############################################################################
# AMOSTRAS
###############################################################################

SAMPLES=(
SRR5221957
SRR5221958
SRR5221961
SRR5221962
)

###############################################################################
# CONFIGURAÇÕES
###############################################################################

THREADS=$(nproc)

BASE_DIR=$(pwd)

echo "==============================================="
echo "Pipeline iniciado"
echo "Data: $(date)"
echo "CPUs disponíveis: $THREADS"
echo "Diretório: $BASE_DIR"
echo "==============================================="

###############################################################################
# DIRETÓRIOS
###############################################################################

mkdir -p input
mkdir -p fastq
mkdir -p trimmed
mkdir -p qc
mkdir -p logs

###############################################################################
# DOWNLOAD + FASTQ + FASTQC
###############################################################################

for SAMPLE in "${SAMPLES[@]}"
do

echo
echo "======================================"
echo "Processando $SAMPLE"
echo "======================================"

mkdir -p qc/$SAMPLE

echo "Download do SRA..."

prefetch "$SAMPLE" \
    --output-directory input

echo "Convertendo para FASTQ..."

fasterq-dump \
    input/$SAMPLE/$SAMPLE.sra \
    --threads $THREADS \
    --outdir fastq

echo "Executando FastQC..."

fastqc \
    fastq/${SAMPLE}.fastq \
    -t $THREADS \
    -o qc/$SAMPLE

done

###############################################################################
# MULTIQC
###############################################################################

echo
echo "Executando MultiQC..."

multiqc qc \
    --outdir qc

###############################################################################
# TRIM GALORE
###############################################################################

echo
echo "Trim Galore..."

for SAMPLE in "${SAMPLES[@]}"
do

trim_galore \
    fastq/${SAMPLE}.fastq \
    --output_dir trimmed

done

###############################################################################
# FASTQC PÓS-TRIMAGEM
###############################################################################

mkdir -p qc/fastqc_trimmed

for SAMPLE in "${SAMPLES[@]}"
do

fastqc \
    trimmed/${SAMPLE}_trimmed.fq \
    -t $THREADS \
    -o qc/fastqc_trimmed

done

###############################################################################
# MULTIQC FINAL
###############################################################################

echo
echo "MultiQC pós-trimagem..."

multiqc \
    qc/fastqc_trimmed \
    --outdir qc \
    --filename multireport_trimmed

echo
echo "======================================="
echo "PIPELINE FINALIZADO"
echo "======================================="

Writing pipeline.sh


In [ ]:
!chmod +x pipeline.sh

In [ ]:
!./pipeline.sh

Pipeline iniciado
Data: Wed Aug 12 04:29:36 PM UTC 2026
CPUs disponíveis: 2
Diretório: /content

Processando SRR5221957
Download do SRA...

2026-08-12T16:29:37 prefetch.2.11.3: Current preference is set to retrieve SRA Normalized Format files with full base quality scores.
2026-08-12T16:29:37 prefetch.2.11.3: 1) Downloading 'SRR5221957'...
2026-08-12T16:29:37 prefetch.2.11.3: SRA Normalized Format file is being retrieved, if this is different from your preference, it may be due to current file availability.
2026-08-12T16:29:37 prefetch.2.11.3:  Downloading via HTTPS...
2026-08-12T16:29:44 prefetch.2.11.3:  HTTPS download succeed
2026-08-12T16:29:47 prefetch.2.11.3:  'SRR5221957' is valid
2026-08-12T16:29:47 prefetch.2.11.3: 1) 'SRR5221957' was downloaded successfully
Convertendo para FASTQ...
spots read      : 9,081,325
reads read      : 9,081,325
reads written   : 9,081,325
Executando FastQC...
[0.028s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to 

In [ ]:
!find . -name "multiqc_report.html"

./qc/multiqc_report.html


In [ ]:
from google.colab import files

files.download("qc/multiqc_report.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!find . -name "*.html"

./qc/SRR5221958/SRR5221958_fastqc.html
./qc/SRR5221961/SRR5221961_fastqc.html
./qc/SRR5221957/SRR5221957_fastqc.html
./qc/SRR5221962/SRR5221962_fastqc.html
./qc/multiqc_report.html


In [ ]:
from google.colab import files

files.download("qc/SRR5221958/SRR5221958_fastqc.html")
files.download("qc/SRR5221961/SRR5221961_fastqc.html")
files.download("qc/SRR5221957/SRR5221957_fastqc.html")
files.download("qc/SRR5221962/SRR5221962_fastqc.html")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!ls qc

multiqc_data	     SRR5221957  SRR5221961
multiqc_report.html  SRR5221958  SRR5221962
